In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prd","qa"],label = "select Environment")
env = dbutils.widgets.get("environment")
# print(env)


silverTablName = f"saleslake_{env}.silver_{env}.cleanedsales"
print("silverTablName:", silverTablName)
goldTablName = f"saleslake_{env}.gold_{env}.refinedsales"
print("goldTablName:",goldTablName )


In [0]:
# Note: The gold table schema expects denormalized columns (product, category, price, region)
# but the silver table has foreign keys (product_id, store_id, customer_id, region_id).
# This placeholder query uses NULL for missing dimension data.
# You need to add JOINs to dimension tables to properly populate product, category, price, and region.

spark.sql(f"""
MERGE INTO {goldTablName} tgt
USING (
    WITH latest_inv_silver AS (
        -- Step 1: Filter only new/updated silver records since last gold load
        SELECT *
        FROM {silverTablName}
        WHERE ingest_ts > (
            SELECT COALESCE(MAX(last_updt_ts), TO_TIMESTAMP('1990-01-01', 'yyyy-MM-dd'))
            FROM {goldTablName}
        )
    ),

    latest_rm_dup_silver AS (
        -- Step 2: Deduplicate — keep latest record per sale_id
        SELECT * FROM (
            SELECT *,
                   ROW_NUMBER() OVER (PARTITION BY sale_id ORDER BY ingest_ts DESC) AS rn
            FROM latest_inv_silver
        ) WHERE rn = 1
    )

    SELECT
        sale_id,
        CAST(NULL AS STRING) AS product,
        CAST(NULL AS STRING) AS category,
        quantity,
        CAST(unit_price AS DOUBLE) AS price,
        sale_date,
        CAST(NULL AS STRING) AS region,
        ingest_ts
    FROM latest_rm_dup_silver

) src
ON tgt.sale_id = src.sale_id

WHEN MATCHED THEN UPDATE SET
    tgt.product = src.product,
    tgt.category = src.category,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.sale_date = src.sale_date,
    tgt.region = src.region,
    tgt.last_updt_ts = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
    sale_id, product, category, quantity, price,
    sale_date, region, initial_load_ts, last_updt_ts
)
VALUES (
    src.sale_id, src.product, src.category, src.quantity, src.price,
    src.sale_date, src.region, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()
)
""")

In [0]:
%sql
SELECT count(*) FROM saleslake_dev.gold_dev.refinedsales;

--SELECT * FROM saleslake_dev.silver_dev.cleanedsales;
